# Population Profiling: 2021–2025

**Status: planned profiling checks completed.** This notebook reads `pondolens.dev_bronze.population`; it does not change Bronze or create Silver tables.

We check whether the data is complete, contains valid values, has duplicates, and whether published totals agree with their components.

One row represents a **geographic area × age group × sex × year × reference date**. The source identifies the values as PSA 2020 census-based population projections. They represent July 1 of each year, rather than observed census counts for each year.

## Inspect the columns

Bronze preserves source values as strings. Values such as `2021.0` and `5039500.0` need numeric conversion in Silver. NULL age limits are expected for `Total`; `85 and over` has no upper age limit.


In [0]:
%sql
DESCRIBE TABLE pondolens.dev_bronze.population;

## Preview source records

Displays 20 sample rows. The preview is not limited to 2021–2025; all profiling checks below are.


In [0]:
%sql
SELECT *
FROM pondolens.dev_bronze.population
LIMIT 20;

## 1. Year coverage

Checks the years, row counts, geographic labels, age groups, sex categories, and reference dates.

**Result:** all five years have 1,026 rows. Each has 18 geographic labels, 19 age groups, and Both/Female/Male. The 18 labels mean **17 regions plus PHILIPPINES**, not 18 regional records. There is no separate NIR label.


In [0]:
%sql
SELECT
    CAST(TRY_CAST(year AS DECIMAL(10,2)) AS INT) AS year,
    COUNT(*) AS rows,
    COUNT(DISTINCT region_name) AS geographic_labels,
    COUNT(DISTINCT age_group) AS age_groups,
    SORT_ARRAY(COLLECT_SET(sex)) AS sex_values,
    SORT_ARRAY(COLLECT_SET(reference_date)) AS reference_dates
FROM pondolens.dev_bronze.population
WHERE TRY_CAST(year AS DECIMAL(10,2)) BETWEEN 2021 AND 2025
GROUP BY 1
ORDER BY 1;

## 2. Missing and invalid values

Checks required fields, numeric population values, reference dates, and rescued rows.

**Result:** 5,130 rows; all reported issue counts are zero. Population values are positive whole numbers. This check intentionally does not flag missing age limits that are valid for Total or 85+.


In [0]:
%sql
WITH scoped AS (
    SELECT *,
        TRY_CAST(population AS DECIMAL(20,2)) AS population_num
    FROM pondolens.dev_bronze.population
    WHERE TRY_CAST(year AS DECIMAL(10,2)) BETWEEN 2021 AND 2025
)
SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(region_name IS NULL OR TRIM(region_name) = '') AS missing_region,
    COUNT_IF(age_group IS NULL OR TRIM(age_group) = '') AS missing_age_group,
    COUNT_IF(sex IS NULL OR TRIM(sex) = '') AS missing_sex,
    COUNT_IF(population IS NULL OR TRIM(population) = '') AS missing_population,
    COUNT_IF(
        population IS NOT NULL AND TRIM(population) <> ''
        AND population_num IS NULL
    ) AS nonnumeric_population,
    COUNT_IF(population_num < 0) AS negative_population,
    COUNT_IF(population_num = 0) AS zero_population,
    COUNT_IF(population_num <> FLOOR(population_num)) AS fractional_population,
    COUNT_IF(TRY_CAST(reference_date AS TIMESTAMP) IS NULL) AS invalid_reference_date,
    COUNT_IF(
        YEAR(TRY_CAST(reference_date AS TIMESTAMP))
        <> CAST(TRY_CAST(year AS DECIMAL(10,2)) AS INT)
    ) AS reference_year_mismatch,
    COUNT_IF(source IS NULL OR TRIM(source) = '') AS missing_source,
    COUNT_IF(_rescued_data IS NOT NULL) AS rescued_rows
FROM scoped;

## 3. Duplicate records

Checks whether more than one row has the same area, age group, sex, year, and reference date.

**Result:** no rows returned, so no duplicate keys were found in this scope.


In [0]:
%sql
SELECT
    region_name,
    age_group,
    sex,
    CAST(TRY_CAST(year AS DECIMAL(10,2)) AS INT) AS year,
    reference_date,
    COUNT(*) AS occurrences
FROM pondolens.dev_bronze.population
WHERE TRY_CAST(year AS DECIMAL(10,2)) BETWEEN 2021 AND 2025
GROUP BY 1, 2, 3, 4, 5
HAVING COUNT(*) > 1
ORDER BY year, region_name, age_group, sex;

## 4. Published population totals

Selects `Total` age group and `Both` sexes. Use this slice as the population denominator for budget per capita; adding Total to the age bands, or Both to Male and Female, would double count.

**Result:** 90 records: 85 regional records and 5 national records. Exclude PHILIPPINES when working at regional level.


In [0]:
%sql
SELECT
    region_name,
    CAST(TRY_CAST(year AS DECIMAL(10,2)) AS INT) AS year,
    reference_date,
    TRY_CAST(population AS DECIMAL(20,2)) AS population
FROM pondolens.dev_bronze.population
WHERE TRY_CAST(year AS DECIMAL(10,2)) BETWEEN 2021 AND 2025
  AND LOWER(TRIM(age_group)) = 'total'
  AND LOWER(TRIM(sex)) = 'both'
ORDER BY region_name, year;

## 5. National total versus regional sum

Compares the published PHILIPPINES total with the sum of the 17 regional totals. Difference means national total minus regional sum.

**Result:** exact agreement in 2021 and 2025. Regional sums are 100 persons higher in 2022–2024. Keep the published values and record the differences.


In [0]:
%sql
WITH totals AS (
    SELECT
        CAST(TRY_CAST(year AS DECIMAL(10,2)) AS INT) AS year,
        UPPER(TRIM(region_name)) AS region_name,
        TRY_CAST(population AS DECIMAL(20,2)) AS population
    FROM pondolens.dev_bronze.population
    WHERE TRY_CAST(year AS DECIMAL(10,2)) BETWEEN 2021 AND 2025
      AND LOWER(TRIM(age_group)) = 'total'
      AND LOWER(TRIM(sex)) = 'both'
),
comparison AS (
    SELECT
        year,
        COUNT_IF(region_name <> 'PHILIPPINES') AS regional_rows,
        MAX(CASE WHEN region_name = 'PHILIPPINES'
                 THEN population END) AS national_population,
        SUM(CASE WHEN region_name <> 'PHILIPPINES'
                 THEN population END) AS regional_population_sum
    FROM totals
    GROUP BY year
)
SELECT *,
    national_population - regional_population_sum AS difference
FROM comparison
ORDER BY year;

## 6. Male plus Female versus Both

Checks that every area/age/year/date combination has all three sex categories, then compares Male + Female with Both.

**Result:** all 1,710 combinations are complete. 386 differ from the published Both value; the largest absolute difference is 100 persons.


In [0]:
%sql
WITH scoped AS (
    SELECT
        region_name,
        age_group,
        CAST(TRY_CAST(year AS DECIMAL(10,2)) AS INT) AS year,
        reference_date,
        LOWER(TRIM(sex)) AS sex,
        TRY_CAST(population AS DECIMAL(20,2)) AS population
    FROM pondolens.dev_bronze.population
    WHERE TRY_CAST(year AS DECIMAL(10,2)) BETWEEN 2021 AND 2025
),
comparison AS (
    SELECT
        region_name,
        age_group,
        year,
        reference_date,
        COUNT_IF(sex = 'male') AS male_rows,
        COUNT_IF(sex = 'female') AS female_rows,
        COUNT_IF(sex = 'both') AS both_rows,
        MAX(CASE WHEN sex = 'male' THEN population END) AS male_population,
        MAX(CASE WHEN sex = 'female' THEN population END) AS female_population,
        MAX(CASE WHEN sex = 'both' THEN population END) AS both_population
    FROM scoped
    GROUP BY region_name, age_group, year, reference_date
)
SELECT
    year,
    COUNT(*) AS combinations_checked,
    COUNT_IF(
        male_rows <> 1 OR female_rows <> 1 OR both_rows <> 1
    ) AS incomplete_combinations,
    COUNT_IF(
        both_population = male_population + female_population
    ) AS exact_matches,
    COUNT_IF(
        both_population <> male_population + female_population
    ) AS differing_combinations,
    MAX(ABS(
        both_population - (male_population + female_population)
    )) AS max_absolute_difference
FROM comparison
GROUP BY year
ORDER BY year;

## 7. Age-band sum versus Total

Checks for one Total row and 18 distinct age bands for every area/sex/year/date combination, then compares the age-band sum with Total.

**Result:** all 270 combinations are complete. 181 differ from Total. Maximum differences by year are 300, 300, 300, 400, and 200 persons for 2021–2025.


In [0]:
%sql
WITH scoped AS (
    SELECT
        region_name,
        sex,
        CAST(TRY_CAST(year AS DECIMAL(10,2)) AS INT) AS year,
        reference_date,
        LOWER(TRIM(age_group)) AS age_group,
        TRY_CAST(population AS DECIMAL(20,2)) AS population
    FROM pondolens.dev_bronze.population
    WHERE TRY_CAST(year AS DECIMAL(10,2)) BETWEEN 2021 AND 2025
),
comparison AS (
    SELECT
        region_name,
        sex,
        year,
        reference_date,
        COUNT_IF(age_group = 'total') AS total_rows,
        COUNT_IF(age_group <> 'total') AS age_group_rows,
        COUNT(DISTINCT CASE WHEN age_group <> 'total'
                            THEN age_group END) AS distinct_age_groups,
        MAX(CASE WHEN age_group = 'total'
                 THEN population END) AS total_population,
        SUM(CASE WHEN age_group <> 'total'
                 THEN population END) AS age_group_sum
    FROM scoped
    GROUP BY region_name, sex, year, reference_date
)
SELECT
    year,
    COUNT(*) AS combinations_checked,
    COUNT_IF(
        total_rows <> 1
        OR age_group_rows <> 18
        OR distinct_age_groups <> 18
    ) AS incomplete_combinations,
    COUNT_IF(total_population = age_group_sum) AS exact_matches,
    COUNT_IF(total_population <> age_group_sum) AS differing_combinations,
    MAX(ABS(total_population - age_group_sum)) AS max_absolute_difference
FROM comparison
GROUP BY year
ORDER BY year;

## Final findings

- **5,130 rows** for 2021–2025; all checked sex and age-band combinations are complete.
- No missing required values, invalid population values, duplicates, or rescued rows.
- Maximum reconciliation differences: **100 persons** for national/regional and sex totals; **400 persons** for age-band totals. Possible rounding is unverified.
- Coverage: **17 regions + PHILIPPINES**.

## Silver preparation

Target: `pondolens.dev_silver.population`

1. Retain all 2021–2025 age and sex breakdowns.
2. Trim text, standardize region names using a reviewed mapping, and preserve `region_name_raw`.
3. Convert columns to the types below. Check whole-number values before casting and retain valid NULL age limits.
4. Assign integer `population_id` values. Preserve existing IDs on reruns.
5. Preserve published population values and source lineage.
6. Verify 5,130 rows, required fields, and key uniqueness after transformation.

### Silver columns

| Column | Type | Meaning |
| --- | --- | --- |
| population_id | BIGINT | Primary key; unique integer |
| region_name | STRING | Standardized geographic name |
| geography_level | STRING | national or regional |
| age_group | STRING | Total or age-band label |
| age_min | INT | Lower age; NULL for Total |
| age_max | INT | Upper age; NULL for Total and 85+ |
| sex | STRING | Both, Female, or Male |
| year | INT | 2021–2025 |
| reference_date | DATE | July 1 reference date |
| population | BIGINT | Published population count |
| source | STRING | Source attribution |
| _drive_file_id | STRING | Drive file ID |
| _source_file | STRING | Source path |
| _source_sheet | STRING | Source worksheet |
| _run_id | STRING | Bronze ingestion run ID |
| _ingested_at | TIMESTAMP | Bronze ingestion time |
| _silver_processed_at | TIMESTAMP | Silver processing time |

**Primary key:** `population_id`

**Business unique key:** `region_name, age_group, sex, year, reference_date`. Include the projection edition if multiple editions are retained.

## Regional analysis

Filter to `age_group = 'Total'`, `sex = 'Both'`, and `geography_level = 'regional'`: **85 rows**.

Align regional boundaries before joining to budget data, especially for NIR. Do not estimate or redistribute NIR population without a verified mapping.

**Budget per capita = budget in PHP ÷ projected population in persons.**

This measures budget per projected resident, not money received by each person.